In [4]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score, 
                           recall_score, f1_score, confusion_matrix)
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

def machine_learning_algorithm_selection_model3():
    """
    Seleção de algoritmos de machine learning para o Modelo 3 (dataset combinado)
    usando nested cross-validation e múltiplos algoritmos para demonstrar
    limitações preditivas mesmo com todas as variáveis disponíveis
    """
    
    print("="*100)
    print("MACHINE LEARNING ALGORITHM SELECTION - MODELO 3 (DATASET COMBINADO)")
    print("="*100)
    
    # Carregar dados do Modelo 3
    model3_train_path = '/Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split/MODEL3TRAIN.csv'
    df_model3 = pd.read_csv(model3_train_path)
    
    # Preparar dados
    target_cols = ['desnutrido', 'eutrofico', 'sobrepeso', 'obeso']
    y_model3 = df_model3['obeso'].copy()
    X_model3 = df_model3.drop(target_cols + ['id_anon'], axis=1)
    
    print(f"MODELO 3 - Dataset Combinado (Demográfico + Alimentação):")
    print(f"  - Observações: {len(df_model3):,}")
    print(f"  - Features: {X_model3.shape[1]}")
    print(f"  - Obesos: {y_model3.sum()} ({y_model3.mean()*100:.1f}%)")
    print(f"  - Não-obesos: {(~y_model3.astype(bool)).sum()} ({(1-y_model3.mean())*100:.1f}%)")
    
    # Configurações de cross-validation
    outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # Definir algoritmos e hiperparâmetros
    algorithms = {
        'Logistic Regression': {
            'model': LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
            'params': {
                'model__C': [0.1, 1.0, 10.0, 100.0],
                'model__penalty': ['l1', 'l2'],
                'model__solver': ['liblinear']
            }
        },
        'Random Forest': {
            'model': RandomForestClassifier(random_state=42, class_weight='balanced'),
            'params': {
                'model__n_estimators': [50, 100, 200],
                'model__max_depth': [3, 5, 10, None],
                'model__min_samples_split': [2, 5, 10]
            }
        },
        'Decision Tree': {
            'model': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
            'params': {
                'model__max_depth': [3, 5, 10, 15, None],
                'model__min_samples_split': [2, 5, 10, 20],
                'model__min_samples_leaf': [1, 2, 5, 10]
            }
        },
        'Gradient Boosting': {
            'model': GradientBoostingClassifier(random_state=42),
            'params': {
                'model__n_estimators': [50, 100, 200],
                'model__learning_rate': [0.01, 0.1, 0.2],
                'model__max_depth': [3, 5, 7]
            }
        },
        'k-Nearest Neighbors': {
            'model': KNeighborsClassifier(),
            'params': {
                'model__n_neighbors': [3, 5, 7, 9, 11],
                'model__weights': ['uniform', 'distance'],
                'model__metric': ['euclidean', 'manhattan']
            }
        },
        'Gaussian Naive Bayes': {
            'model': GaussianNB(),
            'params': {
                'model__var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6]
            }
        },
        'SVM (RBF)': {
            'model': SVC(random_state=42, probability=True, class_weight='balanced'),
            'params': {
                'model__C': [0.1, 1.0, 10.0],
                'model__gamma': ['scale', 'auto', 0.001, 0.01],
                'model__kernel': ['rbf']
            }
        },
        'SVM (Linear)': {
            'model': SVC(random_state=42, probability=True, class_weight='balanced'),
            'params': {
                'model__C': [0.1, 1.0, 10.0, 100.0],
                'model__kernel': ['linear']
            }
        }
    }
    
    def calculate_confidence_intervals(scores, confidence=0.95):
        """Calcula intervalos de confiança usando distribuição t"""
        n = len(scores)
        if n <= 1:
            return np.mean(scores), np.mean(scores), np.mean(scores)
        mean_score = np.mean(scores)
        std_err = stats.sem(scores)
        h = std_err * stats.t.ppf((1 + confidence) / 2., n-1)
        return mean_score, mean_score - h, mean_score + h
    
    def evaluate_algorithm(X_data, y_data, algorithm_name, algorithm_config):
        """Avalia um algoritmo usando nested cross-validation"""
        
        print(f"\n🔍 Avaliando {algorithm_name}...")
        
        # Pipeline com preprocessamento
        pipeline = Pipeline([
            ('scaler', RobustScaler()),
            ('model', algorithm_config['model'])
        ])
        
        # Nested cross-validation
        auc_scores = []
        precision_scores = []
        recall_scores = []
        f1_scores = []
        
        for fold, (train_idx, val_idx) in enumerate(outer_cv.split(X_data, y_data)):
            X_train_fold, X_val_fold = X_data.iloc[train_idx], X_data.iloc[val_idx]
            y_train_fold, y_val_fold = y_data.iloc[train_idx], y_data.iloc[val_idx]
            
            # Grid search no inner CV
            grid_search = GridSearchCV(
                pipeline, algorithm_config['params'], cv=inner_cv,
                scoring='roc_auc', n_jobs=-1, verbose=0
            )
            
            try:
                grid_search.fit(X_train_fold, y_train_fold)
                
                # Predições
                y_pred_proba = grid_search.predict_proba(X_val_fold)[:, 1]
                y_pred = grid_search.predict(X_val_fold)
                
                # Métricas
                auc_scores.append(roc_auc_score(y_val_fold, y_pred_proba))
                precision_scores.append(precision_score(y_val_fold, y_pred, zero_division=0))
                recall_scores.append(recall_score(y_val_fold, y_pred, zero_division=0))
                f1_scores.append(f1_score(y_val_fold, y_pred, zero_division=0))
                
            except Exception as e:
                print(f"    Erro no fold {fold}: {str(e)}")
                # Adicionar scores ruins em caso de erro
                auc_scores.append(0.5)
                precision_scores.append(0.0)
                recall_scores.append(0.0)
                f1_scores.append(0.0)
        
        # Calcular intervalos de confiança
        auc_mean, auc_ci_lower, auc_ci_upper = calculate_confidence_intervals(auc_scores)
        prec_mean, prec_ci_lower, prec_ci_upper = calculate_confidence_intervals(precision_scores)
        rec_mean, rec_ci_lower, rec_ci_upper = calculate_confidence_intervals(recall_scores)
        f1_mean, f1_ci_lower, f1_ci_upper = calculate_confidence_intervals(f1_scores)
        
        return {
            'algorithm': algorithm_name,
            'auc': {'mean': auc_mean, 'ci_lower': auc_ci_lower, 'ci_upper': auc_ci_upper, 'std': np.std(auc_scores)},
            'precision': {'mean': prec_mean, 'ci_lower': prec_ci_lower, 'ci_upper': prec_ci_upper, 'std': np.std(precision_scores)},
            'recall': {'mean': rec_mean, 'ci_lower': rec_ci_lower, 'ci_upper': rec_ci_upper, 'std': np.std(recall_scores)},
            'f1': {'mean': f1_mean, 'ci_lower': f1_ci_lower, 'ci_upper': f1_ci_upper, 'std': np.std(f1_scores)},
            'scores': {'auc': auc_scores, 'precision': precision_scores, 'recall': recall_scores, 'f1': f1_scores}
        }
    
    # =======================================================================
    # AVALIAÇÃO DOS ALGORITMOS
    # =======================================================================
    
    print(f"\n" + "="*80)
    print("AVALIAÇÃO DE ALGORITMOS - MODELO 3")
    print("="*80)
    
    model3_results = []
    for alg_name, alg_config in algorithms.items():
        result = evaluate_algorithm(X_model3, y_model3, alg_name, alg_config)
        model3_results.append(result)
    
    # =======================================================================
    # RESULTADOS COMPARATIVOS
    # =======================================================================
    
    print(f"\n" + "="*100)
    print("RESULTADOS COMPARATIVOS - MODELO 3")
    print("="*100)
    
    print(f"{'Algoritmo':<20} {'AUC-ROC':<25} {'Precision':<25} {'Recall':<25} {'F1-Score':<25}")
    print("-" * 120)
    
    for result in model3_results:
        alg_name = result['algorithm']
        auc = result['auc']
        precision = result['precision']
        recall = result['recall']
        f1 = result['f1']
        
        auc_str = f"{auc['mean']:.3f} ({auc['ci_lower']:.3f}-{auc['ci_upper']:.3f})"
        prec_str = f"{precision['mean']:.3f} ({precision['ci_lower']:.3f}-{precision['ci_upper']:.3f})"
        rec_str = f"{recall['mean']:.3f} ({recall['ci_lower']:.3f}-{recall['ci_upper']:.3f})"
        f1_str = f"{f1['mean']:.3f} ({f1['ci_lower']:.3f}-{f1['ci_upper']:.3f})"
        
        print(f"{alg_name:<20} {auc_str:<25} {prec_str:<25} {rec_str:<25} {f1_str:<25}")
    
    # Identificar melhor algoritmo
    print(f"\n" + "="*80)
    print("MELHOR ALGORITMO - MODELO 3")
    print("="*80)
    
    model3_sorted = sorted(model3_results, key=lambda x: x['auc']['mean'], reverse=True)
    best_model3 = model3_sorted[0]
    
    print(f"MODELO 3 (Dataset Combinado):")
    print(f"  Melhor algoritmo: {best_model3['algorithm']}")
    print(f"  AUC-ROC: {best_model3['auc']['mean']:.3f} (95% CI: {best_model3['auc']['ci_lower']:.3f}-{best_model3['auc']['ci_upper']:.3f})")
    print(f"  Precision: {best_model3['precision']['mean']:.3f}")
    print(f"  Recall: {best_model3['recall']['mean']:.3f}")
    print(f"  F1-Score: {best_model3['f1']['mean']:.3f}")
    
    # Ranking de algoritmos
    print(f"\n" + "="*80)
    print("RANKING DE ALGORITMOS - MODELO 3")
    print("="*80)
    
    print(f"{'Posição':<5} {'Algoritmo':<20} {'AUC-ROC':<15} {'95% CI':<20} {'Precision':<12}")
    print("-" * 75)
    
    for i, result in enumerate(model3_sorted, 1):
        ci_str = f"({result['auc']['ci_lower']:.3f}-{result['auc']['ci_upper']:.3f})"
        prec = result['precision']['mean']
        print(f"{i:<5} {result['algorithm']:<20} {result['auc']['mean']:<15.3f} {ci_str:<20} {prec:<12.3f}")
    
    # Comparação com modelos anteriores
    print(f"\n" + "="*80)
    print("COMPARAÇÃO FINAL ENTRE TODOS OS MODELOS")
    print("="*80)
    
    print(f"RESULTADOS DE MACHINE LEARNING:")
    
    print(f"\nModelo 1 (Demográficas/Perinatais):")
    print(f"  - Melhor algoritmo: Random Forest")
    print(f"  - AUC-ROC: 0.570")
    print(f"  - Features: 27 variáveis")
    print(f"  - Amostra: 6,588 observações")
    
    print(f"\nModelo 2 (Práticas Alimentares):")
    print(f"  - Melhor algoritmo: Logistic Regression")
    print(f"  - AUC-ROC: 0.598")
    print(f"  - Features: 17 variáveis")
    print(f"  - Amostra: 4,510 observações")
    
    print(f"\nModelo 3 (Combinado):")
    print(f"  - Melhor algoritmo: {best_model3['algorithm']}")
    print(f"  - AUC-ROC: {best_model3['auc']['mean']:.3f}")
    print(f"  - Features: {X_model3.shape[1]} variáveis")
    print(f"  - Amostra: {len(df_model3):,} observações")
    
    # Análise final
    print(f"\n" + "="*80)
    print("EVIDÊNCIA DE LIMITAÇÕES PREDITIVAS")
    print("="*80)
    
    modelo2_auc = 0.598
    modelo1_auc = 0.570
    modelo3_auc = best_model3['auc']['mean']
    
    print(f"Progressão dos resultados:")
    print(f"  - Modelo 1: AUC = {modelo1_auc:.3f}")
    print(f"  - Modelo 2: AUC = {modelo2_auc:.3f} (+{modelo2_auc - modelo1_auc:.3f})")
    print(f"  - Modelo 3: AUC = {modelo3_auc:.3f} ({modelo3_auc - modelo2_auc:+.3f})")
    
    if modelo3_auc <= modelo2_auc:
        print(f"\n🔍 CONCLUSÃO CIENTÍFICA:")
        print(f"  - Modelo combinado (44 features) NÃO supera práticas alimentares isoladas")
        print(f"  - Evidência de plateau na capacidade preditiva (~0.6 AUC-ROC)")
        print(f"  - Adicionar variáveis demográficas pode introduzir ruído")
        print(f"  - Limitação inerente dos fatores dos primeiros 24 meses")
    
    print(f"\nLimitações consistentes across algoritmos:")
    precision_range = [r['precision']['mean'] for r in model3_results]
    print(f"  - Precision: {min(precision_range):.3f} - {max(precision_range):.3f}")
    print(f"  - Todos os algoritmos apresentam baixa precision (~3-4%)")
    print(f"  - 96-97% de falsos positivos em qualquer abordagem")
    print(f"  - Capacidade preditiva clinicamente inútil")
    
    print(f"\nImplicações para políticas públicas:")
    print(f"  - Fatores dos primeiros 24 meses têm associação fraca com obesidade aos 2-4 anos")
    print(f"  - Intervenções baseadas nesses preditores seriam ineficazes")
    print(f"  - Necessário focar em fatores proximais (hábitos familiares atuais)")
    print(f"  - Paradigma de predição precoce pode ser inadequado")
    
    return {
        'model3_results': model3_results,
        'best_model3': best_model3,
        'comparative_analysis': {
            'model1_auc': modelo1_auc,
            'model2_auc': modelo2_auc,
            'model3_auc': modelo3_auc
        }
    }

# Executar seleção de algoritmos para Modelo 3
if __name__ == "__main__":
    results = machine_learning_algorithm_selection_model3()

MACHINE LEARNING ALGORITHM SELECTION - MODELO 3 (DATASET COMBINADO)
MODELO 3 - Dataset Combinado (Demográfico + Alimentação):
  - Observações: 4,510
  - Features: 44
  - Obesos: 134 (3.0%)
  - Não-obesos: 4376 (97.0%)

AVALIAÇÃO DE ALGORITMOS - MODELO 3

🔍 Avaliando Logistic Regression...

🔍 Avaliando Random Forest...

🔍 Avaliando Decision Tree...

🔍 Avaliando Gradient Boosting...

🔍 Avaliando k-Nearest Neighbors...

🔍 Avaliando Gaussian Naive Bayes...

🔍 Avaliando SVM (RBF)...

🔍 Avaliando SVM (Linear)...

RESULTADOS COMPARATIVOS - MODELO 3
Algoritmo            AUC-ROC                   Precision                 Recall                    F1-Score                 
------------------------------------------------------------------------------------------------------------------------
Logistic Regression  0.558 (0.481-0.634)       0.038 (0.028-0.048)       0.477 (0.395-0.559)       0.071 (0.053-0.088)      
Random Forest        0.542 (0.474-0.610)       0.017 (-0.014-0.047)      0.098 (-